In [1]:
import numpy as np
import pandas as pd
from types import resolve_bases
import pickle
import plotly.express as px
from SamplingMethods import Sampler_class
from ax.api.client import Client

In [2]:
client = Client()
client = client.load_from_json_file("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/Emulation/StoichModelGP/ModelGP_M15.json")
client.get_next_trials(max_trials=1)

{56: {'n_ci': 0.7747112908656018, 'n_it': 0.39426088961486344}}

In [3]:
def PredictorsToCaStoichs(s1,b1):
    return (0.5+(2.0-0.5)*s1)*b1

def PredictorsToIaStoichs(s2,b1):
    return (1.0+(2.0-1.0)*s2)*(1-b1)

In [4]:
trials = 700

X_lis = []
for i in range(trials):
    sampler = Sampler_class()
    Parameters_lis = [
        {"name":"x1", "type":"range","bounds":[0,1],"value_type":"float"},
        {"name":"x2", "type":"range","bounds":[0,1],"value_type":"float"},
        {"name":"x3", "type":"range","bounds":[0,1],"value_type":"float"}
    ]
    for i in range(100):
        X = sampler.three.QuasirandomSampler3D_func(27,Parameters_lis).T
        if np.shape(X) != (0,3):
            break
    X_lis.append(X)

y_max_lis = []
for X in X_lis:
    s1_arr = X.T[0]
    s2_arr = X.T[1]
    b1_arr = X.T[2]
    n_ci_lis = []
    for s1,b1 in zip(s1_arr,b1_arr):
        n_ci_lis.append(PredictorsToCaStoichs(s1,b1))
    n_it_lis = []
    for s2,b1 in zip(s2_arr,b1_arr):
        n_it_lis.append(PredictorsToIaStoichs(s2,b1))
    n_ci_arr = np.array(n_ci_lis)
    n_it_arr = np.array(n_it_lis)
    y_pred_lis = []
    for n_ci,n_it in zip(n_ci_arr,n_it_arr):
        y_pred_lis.append(client.predict([{"n_ci":n_ci,"n_it":n_it}])[0]["t1"][0])
    y_max_lis.append(np.max(np.array(y_pred_lis)))

y_max_arr = np.array(y_max_lis)
print(y_max_arr.tolist())
print(np.average(y_max_arr))

[14.391900974778299, 13.285866332829883, 13.72306846360765, 14.505966229705002, 13.869044611806721, 17.9989543740296, 13.549573113547883, 14.330124651756652, 14.488356904189587, 16.25014633799135, 13.835485755071419, 13.598304594712422, 15.566212865312075, 13.673325055275706, 14.751119517962884, 13.876154841928022, 14.94794728799218, 13.779687408379745, 16.36823620304832, 15.34919128048523, 15.002796489969336, 14.806975732969962, 17.377091204558095, 14.2637320463083, 18.054163806118407, 16.256229732746274, 14.816473403449606, 15.076979167151581, 13.655126332035644, 15.355500285649857, 14.16486645096693, 17.84040397108675, 13.709922656194644, 14.610099719619694, 13.511196531561158, 14.401022638174137, 17.07571598403432, 14.326245717278947, 15.960558153960248, 14.485663952262643, 13.866579084771514, 15.442883010306304, 13.726975130056141, 17.43051572351752, 16.027533324951776, 13.595627747767459, 15.692722465636791, 15.41965235718596, 17.482092429708505, 14.560011299667028, 14.1946317330

In [5]:
np.average(y_max_arr)

np.float64(15.057531882163241)

In [6]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/Emulation/TestStoichModelGP_M15/DataGenerated/quasirandom_27.pkl"
latestdf = pd.DataFrame(y_max_arr)
pd.to_pickle(obj=latestdf,filepath_or_buffer=filepath)